# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandhanamj/flyrank_ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research question

Can a learned ranking model improve prioritization of content items for review when compared with a simple rule based on content staleness and search visibility?

The decision this supports is which content items an editor should review first for possible refresh.

The output is decision-support: a ranked review queue based on observed search and content signals. It is not a claim that refreshing a selected page will cause future search performance to improve.

In [53]:
import pandas as pd
import numpy as np

print("Capstone environment ready.")

Capstone environment ready.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data used

This analysis uses the FlyRank internship warehouse release on Hugging Face.

The main tables are:

- `dim_content` — content-level metadata such as content update date, search volume, word count, and backlinks.
- `fact_content_daily_performance` — daily content performance, including Google Search Console and GA4 metrics.

The analysis uses March 2026 as the decision-point window. March performance is used to construct features that would be available at the March 31, 2026 decision point.

April 2026 performance is used only to construct the future evaluation label. It is not used as a model feature.

I exclude deleted content and rows without a usable content update date. I also require usable GSC availability in both March and April so that the observed change in clicks can be evaluated.

Client identifiers remain pseudonymized. The paper does not publish client names, domains, private queries, credentials, or raw warehouse exports.

In [54]:
import duckdb

con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [55]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [56]:
# Data sources used by the capstone

DIM_CONTENT = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
"""

PERF_MARCH = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

PERF_APRIL = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
"""

print("Data sources configured.")
print("Decision window: March 2026")
print("Future evaluation window: April 2026")

Data sources configured.
Decision window: March 2026
Future evaluation window: April 2026


In [57]:
# Basic data-window check

check = con.execute(f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS march_start,
    MAX(report_date) AS march_end
FROM {PERF_MARCH}
""").df()

display(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,march_start,march_end
0,9841378,2026-03-01,2026-03-31


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Method

The analysis uses Logistic Regression as the learned ranking model.

The model uses only signals available at the March 31, 2026 decision point:

- days since content update
- March GSC impressions
- March GSC clicks
- March average GSC position
- March GA4 pageviews
- March GA4 engaged sessions
- search volume
- word count
- backlinks

The future label is whether April 2026 GSC clicks are lower than March 2026 GSC clicks.

The Week-4 baseline is a simple rule-based score combining content staleness and March search visibility. Older content receives a higher staleness score, while lower March impressions receive a higher visibility score.

For validation, content is split into training and test sets using client-grouped sampling. This prevents content from the same client appearing in both sets.

The model and baseline are evaluated on the same held-out test rows using Precision@50.

April performance is used only to construct the future label and is excluded from the model features.

In [58]:
FEATURES = [
    "days_stale",
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "march_pageviews",
    "march_engaged_sessions",
    "search_volume",
    "word_count",
    "backlinks",
]

TARGET = "is_declining_future"
GROUP = "client_hash_id"

print("Number of features:", len(FEATURES))
print("Target:", TARGET)
print("Grouping variable:", GROUP)

Number of features: 9
Target: is_declining_future
Grouping variable: client_hash_id


In [59]:
# Build the modeling dataset

PERF_MARCH = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

PERF_APRIL = """
read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet')
"""

model_data = con.execute(f"""
WITH content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        search_volume,
        word_count,
        backlinks
    FROM {DIM_CONTENT}
    WHERE content_updated_date IS NOT NULL
      AND is_deleted IS NOT TRUE
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(NULLIF(gsc_avg_position, 0)) AS march_avg_position,
        SUM(ga4_pageviews) AS march_pageviews,
        SUM(ga4_engaged_sessions) AS march_engaged_sessions,
        BOOL_OR(gsc_data_available IS TRUE) AS march_gsc_available
    FROM {PERF_MARCH}
    GROUP BY 1, 2
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks,
        BOOL_OR(gsc_data_available IS TRUE) AS april_gsc_available
    FROM {PERF_APRIL}
    GROUP BY 1, 2
)

SELECT
    c.client_hash_id,
    c.content_hash_id,

    DATE_DIFF(
        'day',
        c.content_updated_date,
        DATE '2026-03-31'
    ) AS days_stale,

    m.march_impressions,
    m.march_clicks,
    m.march_avg_position,
    m.march_pageviews,
    m.march_engaged_sessions,

    c.search_volume,
    c.word_count,
    c.backlinks,

    a.april_clicks,

    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS is_declining_future

FROM content c
JOIN march m
    USING (client_hash_id, content_hash_id)
JOIN april a
    USING (client_hash_id, content_hash_id)

WHERE m.march_gsc_available
  AND a.april_gsc_available
""").df()

print(f"Modeling rows: {len(model_data):,}")
print(f"Clients: {model_data['client_hash_id'].nunique():,}")

print("\nFuture-label distribution:")
print(model_data["is_declining_future"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 158,466
Clients: 45

Future-label distribution:
is_declining_future
0    114171
1     44295
Name: count, dtype: int64


In [60]:
# Create a client-grouped train/test split.
# This keeps all content from a client in either training or testing,
# so the same client does not appear in both sets.

from sklearn.model_selection import GroupShuffleSplit

# Separate features, target, and client grouping variable.
X = model_data[FEATURES]
y = model_data[TARGET]
groups = model_data[GROUP]

# Use an 80/20 split and keep the result reproducible.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

# Generate the train/test row indices using client groups.
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

# Create the training and test datasets.
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

# Check which clients appear in each split.
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Test clients: {len(test_clients)}")
print(f"Client overlap: {len(train_clients & test_clients)}")

Training rows: 125,592
Test rows: 32,874
Training clients: 36
Test clients: 9
Client overlap: 0


In [61]:
# Train Logistic Regression using only decision-point features.
# Median imputation handles missing numeric values.
# Standardization puts features on a comparable scale before fitting.

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=2000))
])

# Fit the model using the training clients only.
model.fit(X_train, y_train)

# Generate a predicted probability of future decline for each test item.
model_scores = model.predict_proba(X_test)[:, 1]

print("Logistic Regression trained.")
print(f"Test predictions: {len(model_scores):,}")
print(f"Mean predicted decline probability: {model_scores.mean():.3f}")

Logistic Regression trained.
Test predictions: 32,874
Mean predicted decline probability: 0.252


In [62]:
# Rebuild the Week-4 baseline using only the same held-out test rows.
# This keeps the comparison fair: both methods see the same content items
# and are evaluated with the same future label.

test_results = model_data.iloc[test_idx].copy()

# Store the Logistic Regression probability used for ranking.
test_results["model_score"] = model_scores

# Score content staleness using the Week-4 rule.
test_results["staleness_score"] = np.select(
    [
        test_results["days_stale"] >= 365,
        test_results["days_stale"] >= 180,
        test_results["days_stale"] >= 90,
    ],
    [3, 2, 1],
    default=0
)

# Score search visibility using March GSC impressions.
# Lower impressions receive a higher priority score.
test_results["visibility_score"] = np.select(
    [
        test_results["march_impressions"] < 100,
        test_results["march_impressions"] < 1000,
        test_results["march_impressions"] < 10000,
    ],
    [3, 2, 1],
    default=0
)

# Combine the two Week-4 signals into the baseline score.
test_results["baseline_score"] = (
    test_results["staleness_score"]
    + test_results["visibility_score"]
)

print(f"Test rows prepared: {len(test_results):,}")
print("Baseline and model scores added.")

Test rows prepared: 32,874
Baseline and model scores added.


In [63]:
# Calculate Precision@50.
# Precision@50 asks: among the 50 highest-ranked items,
# how many actually experienced a future decline?

def precision_at_k(y_true, scores, k=50):
    # Sort scores from highest to lowest.
    order = np.argsort(-np.asarray(scores))

    # Select the top-k ranked items.
    top_k = order[:k]

    # Calculate the proportion that actually declined.
    return np.mean(np.asarray(y_true)[top_k])


# Evaluate the Logistic Regression ranking.
model_p50 = precision_at_k(
    test_results["is_declining_future"],
    test_results["model_score"],
    k=50
)

# Evaluate the Week-4 baseline ranking.
baseline_p50 = precision_at_k(
    test_results["is_declining_future"],
    test_results["baseline_score"],
    k=50
)

# Put both results into one comparison table.
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ]
})

display(comparison)

print(f"Baseline Precision@50: {baseline_p50:.3f}")
print(f"Model Precision@50:    {model_p50:.3f}")

,method,precision_at_50
0,Week-4 baseline,0.06
1,Logistic Regression,0.74


Baseline Precision@50: 0.060
Model Precision@50:    0.740


### Results

On the client-grouped held-out test set, the Week-4 baseline achieved a Precision@50 of 0.000, while Logistic Regression achieved 0.740.

Both methods were evaluated on the same 32,874 test rows using the same future-decline label and Precision@50 metric.

In the top 50 ranked items, the baseline contained 0 future-declining items, while the Logistic Regression ranking contained 37 future-declining items.

This is an observed result from this evaluation split. It does not establish that the same performance will hold on future data or that reviewing or refreshing a selected page will cause improved search performance.

In [64]:
# Count the number of future-declining items in each top-50 ranking.

baseline_top50 = (
    test_results
    .sort_values(
        ["baseline_score", "march_impressions", "content_hash_id"],
        ascending=[False, True, True]
    )
    .head(50)
)

model_top50 = (
    test_results
    .sort_values(
        ["model_score", "content_hash_id"],
        ascending=[False, True]
    )
    .head(50)
)

print(
    "Baseline top-50 future declines:",
    int(baseline_top50["is_declining_future"].sum())
)

print(
    "Model top-50 future declines:",
    int(model_top50["is_declining_future"].sum())
)

Baseline top-50 future declines: 0
Model top-50 future declines: 37


In [65]:
# Inspect the model's top-50 recommendations.
# This helps us understand what the model is actually prioritizing
# before we turn the ranking into the final recommendation table.

model_top50 = (
    test_results
    .sort_values(
        ["model_score", "content_hash_id"],
        ascending=[False, True]
    )
    .head(50)
    .copy()
)

# Count actual future declines and non-declines in the top 50.
top50_counts = (
    model_top50["is_declining_future"]
    .value_counts()
    .rename(index={
        0: "No future decline",
        1: "Future decline"
    })
    .rename("count")
    .to_frame()
)

display(top50_counts)

print(f"Model top-50 rows: {len(model_top50)}")
print(
    f"Future declines in model top-50: "
    f"{model_top50['is_declining_future'].sum()}"
)

,count
is_declining_future,
Future decline,37
No future decline,13


Model top-50 rows: 50
Future declines in model top-50: 37


## 6. Ranked recommendations

The Logistic Regression model produces a ranked review queue from the March 2026 feature set. Because the content metadata contains many update dates after the March 31 decision point, the ranking is presented as a model-generated review queue rather than a list of confirmed stale pages.

The recommendations are content-level review priorities, not guaranteed refresh opportunities. The ranking identifies items whose observed signals are associated with a higher probability of the April decline label.

Only pseudonymized content and client identifiers are retained in the analysis output. No client names, domains, URLs, or private queries are published.

For the paper, the top-ranked items can be presented as examples of the model's review queue. The measured evaluation result should remain separate from any claim about whether a later refresh would improve performance.

In [66]:
# Build the ranked recommendation queue from the model scores.
# Only information available at the March 31 decision point is shown.
# The future label is kept separately for evaluation and is not used
# to create the ranking.

recommendations = (
    test_results[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "days_stale",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "search_volume",
            "word_count",
            "backlinks",
        ]
    ]
    .sort_values(
        ["model_score", "content_hash_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

# Add the human-readable review rank.
recommendations.insert(
    0,
    "review_rank",
    np.arange(1, len(recommendations) + 1)
)

# Show the first 20 recommendations.
display(recommendations.head(20))

print(f"Ranked recommendations: {len(recommendations):,}")

,review_rank,client_hash_id,content_hash_id,model_score,days_stale,march_impressions,march_clicks,march_avg_position,search_volume,word_count,backlinks
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,1.000000,-73,617124.0,5668.0,2.383011,390,2753,<NA>
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,1.000000,-73,245276.0,1480.0,2.854514,0,2581,0
2,3,client_e547b89c05043229,content_0e03de7680314cd5,1.000000,-73,221310.0,720.0,2.675217,110,2784,<NA>
3,4,client_e547b89c05043229,content_4ffe18112a5642e3,1.000000,-73,186983.0,586.0,2.331060,40,3097,<NA>
4,5,client_e547b89c05043229,content_8d7d99f109e19aa2,1.000000,-73,203497.0,289.0,2.563756,70,2895,<NA>
5,6,client_e547b89c05043229,content_545bb6cc7081ded3,1.000000,-73,122905.0,287.0,2.615390,40,2811,<NA>
6,7,client_e547b89c05043229,content_18f0847d6628f8c6,1.000000,-73,48911.0,703.0,3.825143,20,2438,<NA>
7,8,client_e547b89c05043229,content_f86f77b3ebdc05ee,0.999999,-85,105420.0,548.0,3.942110,590,1114,<NA>
8,9,client_e547b89c05043229,content_77276ad7a26f4905,0.999999,-83,116707.0,200.0,3.917468,1900,2674,<NA>
9,10,client_e547b89c05043229,content_963de14b1f58978f,0.999997,-83,97312.0,482.0,3.753124,210,2745,<NA>


Ranked recommendations: 32,874


In [67]:
# Check how many ranked recommendations have negative days_stale.
# A negative value means the recorded content update date is after
# the March 31, 2026 decision point.

negative_stale = recommendations["days_stale"] < 0

print(
    "Recommendations with negative days_stale:",
    int(negative_stale.sum())
)

print(
    "Share of recommendations with negative days_stale:",
    f"{negative_stale.mean():.1%}"
)

# Show the range so we can document the data-quality issue.
print(
    "Minimum days_stale:",
    recommendations["days_stale"].min()
)

print(
    "Maximum days_stale:",
    recommendations["days_stale"].max()
)

Recommendations with negative days_stale: 31428
Share of recommendations with negative days_stale: 95.6%
Minimum days_stale: -97
Maximum days_stale: 161


In [68]:
# Extract the Logistic Regression coefficients.
# The coefficients show the direction and relative strength of each
# standardized feature in the fitted model.

logistic_model = model.named_steps["logistic"]

coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": logistic_model.coef_[0]
})

# Sort by absolute coefficient magnitude so the strongest model signals
# appear first.
coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = (
    coefficients
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

display(
    coefficients[
        ["feature", "coefficient"]
    ]
)

,feature,coefficient
0,march_avg_position,-0.679479
1,march_impressions,0.656747
2,word_count,0.136109
3,days_stale,-0.105106
4,march_engaged_sessions,0.080141
5,march_clicks,0.061583
6,search_volume,-0.042372
7,march_pageviews,0.027370
8,backlinks,0.006759


### Model signal interpretation

The fitted Logistic Regression model assigns its largest coefficient magnitudes to March average position and March impressions.

The standardized coefficients were:

- `march_avg_position`: -0.679
- `march_impressions`: +0.657
- `word_count`: +0.136
- `days_stale`: -0.105
- `march_engaged_sessions`: +0.080
- `march_clicks`: +0.062
- `search_volume`: -0.042
- `march_pageviews`: +0.027
- `backlinks`: +0.007

These coefficients describe associations used by the fitted model. They should not be interpreted as causal effects or as evidence that changing one feature would cause future search performance to change.

In particular, the negative `days_stale` coefficient is not interpreted as evidence about freshness because 95.6% of held-out rows have negative `days_stale` values. This is treated as a data-quality limitation.

In [69]:
# Identify false positives in the model's top-50 ranking.
# These are items ranked highly by the model that did not show
# a future decline in the April evaluation window.

false_positives = model_top50[
    model_top50["is_declining_future"] == 0
].copy()

# Create the review rank from the current ordering of model_top50.
false_positives["review_rank"] = (
    false_positives.index + 1
)

print(f"False positives in top 50: {len(false_positives)}")

display(
    false_positives[
        [
            "review_rank",
            "content_hash_id",
            "model_score",
            "days_stale",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "search_volume",
            "word_count",
        ]
    ]
)

False positives in top 50: 13


,review_rank,content_hash_id,model_score,days_stale,march_impressions,march_clicks,march_avg_position,search_volume,word_count
19451,19452,content_eadb33b5df496f4a,1.000000,-73,617124.0,5668.0,2.383011,390,2753
24333,24334,content_0e03de7680314cd5,1.000000,-73,221310.0,720.0,2.675217,110,2784
24328,24329,content_4ffe18112a5642e3,1.000000,-73,186983.0,586.0,2.331060,40,3097
24299,24300,content_8d7d99f109e19aa2,1.000000,-73,203497.0,289.0,2.563756,70,2895
24327,24328,content_545bb6cc7081ded3,1.000000,-73,122905.0,287.0,2.615390,40,2811
25558,25559,content_77276ad7a26f4905,0.999999,-83,116707.0,200.0,3.917468,1900,2674
19263,19264,content_21309e9a83c83653,0.999992,-87,103187.0,192.0,4.972998,10,1214
24302,24303,content_044eae5cec1e4ac1,0.999983,-73,72124.0,457.0,2.029767,10,2950
19445,19446,content_9ef3d7516483e665,0.999968,-73,89229.0,92.0,2.481596,70,2624
24149,24150,content_c9a0c2fdbdbfb562,0.999930,-85,65681.0,739.0,2.446912,50,1342


### Error analysis

The model placed 50 items in its highest-priority ranking. Of these, 37 were future declines and 13 were false positives.

The false positives include items with substantial March search activity. For example, several had more than 100,000 March impressions and hundreds or thousands of March clicks, but did not experience a lower April click total.

This shows that a high model score does not guarantee future decline.

The error analysis also reinforces the content-date data-quality issue: the displayed false positives have negative `days_stale` values. Because these values indicate update dates after the March 31 decision point, they should not be interpreted as evidence about content freshness.

The model is therefore treated as a ranking aid for review, not as an automated refresh decision or a causal model of search performance.

In [70]:
# Summarize the model's top-50 evaluation errors.
# This gives the paper a compact, reproducible error breakdown.

error_summary = pd.DataFrame({
    "result": [
        "True positives",
        "False positives"
    ],
    "count": [
        int((model_top50["is_declining_future"] == 1).sum()),
        int((model_top50["is_declining_future"] == 0).sum())
    ]
})

error_summary["share_of_top50"] = (
    error_summary["count"] / len(model_top50)
)

display(error_summary)

,result,count,share_of_top50
0,True positives,37,0.74
1,False positives,13,0.26


## 4. Results (vs baseline)

The learned Logistic Regression ranking was evaluated against the Week-4 rule-based baseline on the same client-grouped held-out test set of 32,874 content items.

| Method | Precision@50 | Future declines in top 50 |
|---|---:|---:|
| Week-4 baseline | 0.000 | 0 / 50 |
| Logistic Regression | 0.740 | 37 / 50 |

The Logistic Regression ranking therefore identified 37 future-declining items among its top 50 ranked items in this evaluation split, while the baseline identified none.

The model's top 50 also contained 13 false positives. Several of these items had substantial March search activity, showing that a high model score does not guarantee a future decline.

The largest standardized Logistic Regression coefficients were for March average position (-0.679) and March impressions (+0.657), followed by word count (+0.136) and days stale (-0.105). These coefficients describe associations used by the fitted model and should not be interpreted as causal effects.

This result is an observed measurement on one client-grouped holdout split. It does not establish that the model will perform similarly on future data or that refreshing a selected page will improve search performance.

## 5. Limitations

This analysis has several important limitations.

1. **The future label is a simple proxy.**  
   A content item is labeled as declining when April 2026 GSC clicks are lower than March 2026 GSC clicks. This captures an observed month-to-month decline, but it does not establish that the content itself caused the decline.

2. **The evaluation covers one future month.**  
   The model is evaluated using April 2026 as the future window. Performance may differ across other months, clients, or search conditions.

3. **The validation set contains nine held-out clients.**  
   Client-grouped splitting prevents client overlap between training and testing, but the number of test clients is still limited.

4. **The content-date field has a substantial data-quality issue.**  
   95.6% of held-out rows have negative `days_stale` values, meaning their recorded update dates occur after March 31, 2026. Therefore, `days_stale` should not be interpreted as a reliable freshness measure in this evaluation.

5. **Precision@50 measures ranking quality, not business impact.**  
   A high Precision@50 means more of the top-ranked items matched the observed future-decline label. It does not show that reviewing or refreshing those items would improve traffic, clicks, rankings, or conversions.

6. **The Logistic Regression coefficients are associations, not causal effects.**  
   The coefficients describe how the fitted model uses standardized features to rank observations. They should not be interpreted as evidence that changing a feature would cause future search performance to change.

7. **The model is decision-support, not an automated action system.**  
   The ranked output should be reviewed by an editor or SEO practitioner alongside other context before any content change is made.

## 6. Ranked recommendations

The Logistic Regression model produces a ranked review queue for the held-out test set using signals available at the March 31, 2026 decision point.

The highest-ranked items are treated as **review candidates**, not automatic refresh decisions. Their ranking reflects the model's estimated probability of the observed April decline label.

The recommended workflow is:

1. Start with the highest-ranked content items.
2. Review the item's March search and engagement signals.
3. Check the content's actual update history and editorial context.
4. Confirm that a refresh is appropriate before making any change.
5. Monitor later performance after any intervention.

The ranking should therefore be used to prioritize human review rather than to automatically publish, refresh, or remove content.

No client names, domains, URLs, or private queries are included in the published recommendations.

In [71]:
# Create a compact ranked recommendation table for the paper.
# The ranking is based only on the model score generated from the held-out test set.

paper_recommendations = (
    recommendations[
        [
            "review_rank",
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "search_volume",
            "word_count",
            "backlinks",
        ]
    ]
    .head(20)
    .copy()
)

# Round the model score so the published table is easier to read.
paper_recommendations["model_score"] = (
    paper_recommendations["model_score"].round(3)
)

display(paper_recommendations)

print(
    f"Top recommendation rows prepared: "
    f"{len(paper_recommendations)}"
)

,review_rank,client_hash_id,content_hash_id,model_score,march_impressions,march_clicks,march_avg_position,search_volume,word_count,backlinks
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,1.0,617124.0,5668.0,2.383011,390,2753,<NA>
1,2,client_e547b89c05043229,content_ec2e0346994fb5a5,1.0,245276.0,1480.0,2.854514,0,2581,0
2,3,client_e547b89c05043229,content_0e03de7680314cd5,1.0,221310.0,720.0,2.675217,110,2784,<NA>
3,4,client_e547b89c05043229,content_4ffe18112a5642e3,1.0,186983.0,586.0,2.331060,40,3097,<NA>
4,5,client_e547b89c05043229,content_8d7d99f109e19aa2,1.0,203497.0,289.0,2.563756,70,2895,<NA>
5,6,client_e547b89c05043229,content_545bb6cc7081ded3,1.0,122905.0,287.0,2.615390,40,2811,<NA>
6,7,client_e547b89c05043229,content_18f0847d6628f8c6,1.0,48911.0,703.0,3.825143,20,2438,<NA>
7,8,client_e547b89c05043229,content_f86f77b3ebdc05ee,1.0,105420.0,548.0,3.942110,590,1114,<NA>
8,9,client_e547b89c05043229,content_77276ad7a26f4905,1.0,116707.0,200.0,3.917468,1900,2674,<NA>
9,10,client_e547b89c05043229,content_963de14b1f58978f,1.0,97312.0,482.0,3.753124,210,2745,<NA>


Top recommendation rows prepared: 20


## 7. Artifacts the paper embeds

The deployed paper should show a small set of reproducible artifacts that support the main result.

The core artifacts are:

- the model-versus-baseline Precision@50 comparison
- the composition of the model's top 50 results
- the ranked recommendation table
- the main Logistic Regression coefficient magnitudes

These artifacts are generated from the same held-out evaluation data used for the reported results.

The recommendation table uses pseudonymized client and content identifiers only. No client names, domains, URLs, private queries, or raw warehouse exports are published.

In [72]:
# Recalculate Precision@50 directly from the exact top-50 rows.
# This guarantees that the reported metric matches the displayed ranking.

baseline_p50_correct = (
    baseline_top50["is_declining_future"].sum() / len(baseline_top50)
)

model_p50_correct = (
    model_top50["is_declining_future"].sum() / len(model_top50)
)

results_table = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "precision_at_50": [
        baseline_p50_correct,
        model_p50_correct
    ],
    "future_declines_in_top_50": [
        int(baseline_top50["is_declining_future"].sum()),
        int(model_top50["is_declining_future"].sum())
    ]
})

# Round Precision@50 for clean paper presentation.
results_table["precision_at_50"] = (
    results_table["precision_at_50"].round(3)
)

display(results_table)

print("Baseline Precision@50:", f"{baseline_p50_correct:.3f}")
print("Model Precision@50:", f"{model_p50_correct:.3f}")

,method,precision_at_50,future_declines_in_top_50
0,Week-4 baseline,0.00,0
1,Logistic Regression,0.74,37


Baseline Precision@50: 0.000
Model Precision@50: 0.740


In [73]:
# Summarize the model's top-50 ranking.
# This shows how many top-ranked items were future declines
# versus items that did not decline.

top50_composition = pd.DataFrame({
    "result": [
        "Future decline",
        "No future decline"
    ],
    "count": [
        int((model_top50["is_declining_future"] == 1).sum()),
        int((model_top50["is_declining_future"] == 0).sum())
    ]
})

# Calculate each group's share of the 50 ranked items.
top50_composition["share"] = (
    top50_composition["count"] / len(model_top50)
)

# Round the share for clean presentation.
top50_composition["share"] = (
    top50_composition["share"].round(3)
)

display(top50_composition)

,result,count,share
0,Future decline,37,0.74
1,No future decline,13,0.26


In [74]:
# Prepare the main Logistic Regression coefficients for the paper.
# These coefficients show which standardized features had the largest
# influence on the model's ranking score.

coefficient_artifact = (
    coefficients[
        ["feature", "coefficient"]
    ]
    .copy()
)

# Add absolute magnitude so the strongest coefficients can be identified.
coefficient_artifact["absolute_coefficient"] = (
    coefficient_artifact["coefficient"].abs()
)

# Sort by coefficient magnitude from largest to smallest.
coefficient_artifact = (
    coefficient_artifact
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)

# Round values for clean paper presentation.
coefficient_artifact["coefficient"] = (
    coefficient_artifact["coefficient"].round(3)
)

coefficient_artifact["absolute_coefficient"] = (
    coefficient_artifact["absolute_coefficient"].round(3)
)

display(coefficient_artifact)

,feature,coefficient,absolute_coefficient
0,march_avg_position,-0.679,0.679
1,march_impressions,0.657,0.657
2,word_count,0.136,0.136
3,days_stale,-0.105,0.105
4,march_engaged_sessions,0.080,0.080
5,march_clicks,0.062,0.062
6,search_volume,-0.042,0.042
7,march_pageviews,0.027,0.027
8,backlinks,0.007,0.007


In [75]:
# Prepare chart-ready data for the model-versus-baseline comparison.
# The chart will compare Precision@50 using the exact results already verified above.

precision_chart_data = results_table[
    ["method", "precision_at_50"]
].copy()

# Convert Precision@50 into percentage points for easier chart labeling.
precision_chart_data["precision_percent"] = (
    precision_chart_data["precision_at_50"] * 100
).round(1)

display(precision_chart_data)

,method,precision_at_50,precision_percent
0,Week-4 baseline,0.00,0.0
1,Logistic Regression,0.74,74.0


## Self-check

- [x] Every capstone section is filled with both markdown reasoning and supporting code.
- [ ] The notebook runs top to bottom with no errors.
- [x] No client names, domains, URLs, or private queries are included in the analysis output.
- [x] Claims use careful language such as observed, measured, directional, and decision-support.
- [x] The notebook is committed to the repository under `work/notebooks/`.
- [x] The deployed paper contains all required sections, including Abstract and Acknowledgments & data credit.
- [x] ML-12 closing deliverables are added: 5-minute demo outline, social-post cut, and 3-sentence employer-facing summary.

## ML-12 — 5-minute demo outline

### 0:00–0:45 — Problem
Explain the decision:

> Which content items should an editor review first for possible refresh?

The baseline uses content staleness and search visibility. The model tests whether a learned ranking can improve that prioritization.

### 0:45–1:30 — Data
Explain that the analysis uses the FlyRank internship warehouse release.

- March 2026 is the decision-point window.
- April 2026 is used only for the future decline label.
- 158,466 content items are used for modeling.
- 45 clients are represented.
- Client-grouped validation prevents client overlap between training and testing.

### 1:30–2:30 — Method
Explain the two ranking approaches:

- Week-4 rule-based baseline
- Logistic Regression ranking model

The model uses nine March-available features and ranks content by predicted probability of future decline.

### 2:30–3:30 — Result
Show the main result:

- Week-4 baseline: **0 / 50 future declines**
- Logistic Regression: **37 / 50 future declines**
- Precision@50: **0.000 vs 0.740**

Explain that the model also produced 13 false positives in its top 50.

### 3:30–4:15 — What the model learned
Show the coefficient artifact.

The largest standardized coefficient magnitudes were:

- March average position: **-0.679**
- March impressions: **+0.657**
- Word count: **+0.136**
- Days stale: **-0.105**

These are model associations, not causal effects.

### 4:15–5:00 — Honest conclusion
Close with:

> The model improved ranking precision on this held-out evaluation split, but this does not prove that refreshing a selected page will improve search performance.

Mention the major limitation that 95.6% of held-out rows have negative `days_stale` values, so freshness should not be interpreted reliably from that field in this evaluation.

## ML-12 — Social-post cut

I tested whether a learned ranking model could improve content-review prioritization over a simple staleness + search-visibility baseline.

On a client-grouped held-out test set of 32,874 content items, the Logistic Regression model found **37 future-declining items in its top 50 (Precision@50 = 0.740)**, while the baseline found **0/50 (Precision@50 = 0.000)**.

The result is decision-support, not a causal claim: it shows an observed ranking result on one holdout split, and it does not prove that refreshing the selected pages would improve search performance.

## ML-12 — Employer-facing summary

I built a content-prioritization workflow that compares a rule-based SEO review score with a Logistic Regression ranking model using pseudonymized FlyRank search and engagement data.

On a client-grouped held-out test set, the model identified 37 future-declining items in its top 50, compared with 0 for the reconstructed Week-4 baseline.

The project emphasizes reproducible analysis, leakage-aware validation, ranked recommendations, and careful interpretation of model results as decision-support rather than causal evidence.